In [1]:
# web 문서를 읽어 RAG 구현
!pip install sentence-transformers chromadb google-generativeai  python-dotenv
!pip install requests beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.5 MB/s et

In [2]:
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb import PersistentClient

In [5]:
# 웹 페이지에서 텍스트 추출
def extract_from_urlFunc(url):
    # headers 주면 403/404 잘 안뜸, 스크래핑 안정성 증가, 서버가 브라우저 요청으로 인식
    headers = { "User-Agent": "Mozilla/5.0" }
    resp = requests.get(url, headers=headers)
    print("status_code:", resp.status_code)

    if resp.status_code != 200:
        print("요청 실패, html preview:", resp.text[:200])
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    paragraphs = soup.find_all("p")

    texts = [
        p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
    ]
    print("found <p> count:", len(texts))
    return texts

# 임베딩 모델 로딩
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ChromaDB 클라이언트 및 컬렉션 생성
chroma_client = PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection("webdata")

# 데이터 추출 및 저장
url = "https://ko.wikipedia.org/wiki/김치찌개"
web_docs = extract_from_urlFunc(url)
print("web_docs 개수:", len(web_docs))

if not web_docs:
    print("추출된 문서가 없습니다. 종료합니다.")
    raise Exception("문서 없음: web_docs가 비어 있음")
else:
    web_embeddings = embedder.encode(web_docs)
    print("web_embeddings shape:", web_embeddings.shape)

    for i, (doc, emb) in enumerate(zip(web_docs, web_embeddings)):
        collection.add (
            documents=[doc],
            embeddings=[emb.tolist()],
            ids=[f"web_{i}"]
        )

    print("총 저장된 문서 수:", collection.count())

    result = collection.get(ids=["web_0", "web_1"])
    print("샘플 문서:", result["documents"])


# 웹에서 얻은 샘플 자료로 LLM에 질문할 prompt 강화
from google import genai
import os
from dotenv import load_dotenv
load_dotenv()
# from google.colab import userdata
# userdata.get('secretName')

query = "김치찌개의 역사와 조리법 알려줘"
query_vec = embedder.encode([query])[0]

results = collection.query (
    query_embeddings=[query_vec.tolist()],
    n_results=3,
    include=["documents", "distances"]
)

retrieved_docs = results["documents"][0]
retrieved_dist = results["distances"][0]
for i, (doc, dist) in enumerate(zip(retrieved_docs, retrieved_dist), 1):
    print(f"문서{i} : {doc}")
    print(f"distance: {dist:.4f}\n")

# 프롬프트
prompt = f"""
  다음 문서를 참고하여 '{query}'에 대해 답변해줘:
  {retrieved_docs}
  요구사항:
  - 김치찌개의 역사, 유래, 지역별 특징, 조리 단계, 재료의 역할까지 설명할 것
  - 최소 20 문장 이내로 작성할 것
  - 마크다운 없이
  - 각 문장을 줄바꿈으로 분리
"""
print("prompt 내용:", prompt)

# 응답 생성
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt
)

print("\n답변 : ")
print(response.text)

status_code: 200
found <p> count: 9
web_docs 개수: 9
web_embeddings shape: (9, 384)
총 저장된 문서 수: 9
샘플 문서: ['국물류', '반찬']
문서1 : 김치찌개는 대표적인한국 요리중 하나로,김치를 넣고 얼큰하게 끓인찌개이다.[1]된장찌개·순두부찌개와 함께 가장 널리 알려진찌개요리이다.
distance: 0.6230

문서2 : 영양적으로는 김치 외에 다른 재료를 넣어 영양소를 보충할 수 있으나, 김치는 높은 온도로 끓일 경우유산균이 파괴되므로 영양적 면이 떨어지게 된다. 그리고 김치와 다른 부가재료 때문에나트륨함량도 매우 높으며, 볶을 때 사용되는 기름 때문에지방함량도 높은 편이다.따라서칼로리도 높은 편이다.
distance: 0.7264

문서3 : 국물류
distance: 0.9596

prompt 내용: 
  다음 문서를 참고하여 '김치찌개의 역사와 조리법 알려줘'에 대해 답변해줘:
  ['김치찌개는 대표적인한국 요리중 하나로,김치를 넣고 얼큰하게 끓인찌개이다.[1]된장찌개·순두부찌개와 함께 가장 널리 알려진찌개요리이다.', '영양적으로는 김치 외에 다른 재료를 넣어 영양소를 보충할 수 있으나, 김치는 높은 온도로 끓일 경우유산균이 파괴되므로 영양적 면이 떨어지게 된다. 그리고 김치와 다른 부가재료 때문에나트륨함량도 매우 높으며, 볶을 때 사용되는 기름 때문에지방함량도 높은 편이다.따라서칼로리도 높은 편이다.', '국물류']
  요구사항:
  - 김치찌개의 역사, 유래, 지역별 특징, 조리 단계, 재료의 역할까지 설명할 것
  - 최소 20 문장 이내로 작성할 것
  - 마크다운 없이
  - 각 문장을 줄바꿈으로 분리



ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 37.613399446s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type'